In [1]:
# install dependencies

import subprocess
subprocess.run(["pip", "install", "-q",
    "openai", "pinecone", "groq",
    "sentence-transformers", "rank-bm25", "scikit-learn"
], check=True)
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 14.3 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# Imports
import os
import pickle
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
from pinecone import Pinecone
from groq import Groq
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from kaggle_secrets import UserSecretsClient

print("✓ Imports done")

✓ Imports done


In [3]:
# Load secrets
secrets       = UserSecretsClient()
OPENAI_KEY    = secrets.get_secret("OPENAI_API_KEY")
PINECONE_KEY  = secrets.get_secret("PINECONE_API_KEY")
GROQ_KEY      = secrets.get_secret("GROQ_API_KEY")

print("✓ Secrets loaded")

✓ Secrets loaded


In [4]:
# Initialize clients
openai_client = OpenAI(api_key=OPENAI_KEY)
pc            = Pinecone(api_key=PINECONE_KEY)
groq_client   = Groq(api_key=GROQ_KEY)
reranker      = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✓ Clients ready")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✓ Clients ready


In [5]:
# ── CELL 5: Load BM25 for all 3 strategies ───────────────────
print("Building BM25 indexes for all 3 strategies...")

BM25_INDEXES = {}
CHUNK_STORES = {}

for label, pkl_name in [
    ("A", "corpus_A_fixed.pkl"),
    ("B", "corpus_B_recursive.pkl"),
    ("C", "corpus_C_semantic.pkl"),
]:
    path = f"/kaggle/input/datasets/rockybhai159/prime123/{pkl_name}"
    with open(path, "rb") as f:
        data = pickle.load(f)

    chunks   = data["chunks"]
    metadata = data["metadata"]

    tokenized            = [chunk.lower().split() for chunk in chunks]
    BM25_INDEXES[label]  = BM25Okapi(tokenized)
    CHUNK_STORES[label]  = {"chunks": chunks, "metadata": metadata}

    print(f"  ✓ BM25 Strategy {label}: {len(chunks)} chunks")

print("✓ All BM25 indexes ready")

Building BM25 indexes for all 3 strategies...
  ✓ BM25 Strategy A: 2007 chunks
  ✓ BM25 Strategy B: 2753 chunks
  ✓ BM25 Strategy C: 1455 chunks
✓ All BM25 indexes ready


In [6]:
# CELL 6: Pinecone index map
INDEXES = {
    "A": pc.Index("rag-index-a-fixed"),
    "B": pc.Index("rag-index-b-recursive"),
    "C": pc.Index("rag-index-c-semantic"),
}
print("✓ Pinecone indexes connected")
for key, idx in INDEXES.items():
    stats = idx.describe_index_stats()
    print(f"   Strategy {key}: {stats['total_vector_count']} vectors")

✓ Pinecone indexes connected
   Strategy A: 2007 vectors
   Strategy B: 2753 vectors
   Strategy C: 1455 vectors


In [10]:
# CORE FUNCTIONS

def embed_query(query):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query.replace("\n", " "),
    )
    return response.data[0].embedding


def rrf_combine(semantic_ids, bm25_ids, k=60):
    scores = {}
    for rank, doc_id in enumerate(semantic_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    for rank, doc_id in enumerate(bm25_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)


def retrieve(query, strategy="B", mode="hybrid", top_k=15, final_k=5):

    index = INDEXES[strategy]

    # Step 1: Semantic search via Pinecone
    t0               = time.time()
    q_vector         = embed_query(query)
    semantic_results = index.query(
        vector=q_vector,
        top_k=top_k,
        include_metadata=True
    )
    retrieval_time = round(time.time() - t0, 3)

    pinecone_chunks = [
        (r["metadata"]["text"], r["metadata"])
        for r in semantic_results["matches"]
        if "text" in r["metadata"]
    ]

    if mode == "semantic":
        return pinecone_chunks[:final_k], retrieval_time

    # Step 2: BM25 — use matching corpus for this strategy
    bm25_index   = BM25_INDEXES[strategy]
    chunk_data   = CHUNK_STORES[strategy]
    bm25_scores  = bm25_index.get_scores(query.lower().split())
    bm25_top_ids = np.argsort(bm25_scores)[::-1][:top_k].tolist()
    bm25_chunks  = [
        (chunk_data["chunks"][i], chunk_data["metadata"][i])
        for i in bm25_top_ids if i < len(chunk_data["chunks"])
    ]

    # Step 3: RRF
    k          = 60
    rrf_scores = {}
    chunk_store = {}

    for rank, (chunk, meta) in enumerate(pinecone_chunks):
        key = chunk[:80]
        rrf_scores[key]  = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_store[key] = (chunk, meta)

    for rank, (chunk, meta) in enumerate(bm25_chunks):
        key = chunk[:80]
        rrf_scores[key]  = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_store[key] = (chunk, meta)

    sorted_keys    = sorted(rrf_scores.keys(),
                            key=lambda x: rrf_scores[x],
                            reverse=True)
    rrf_candidates = [chunk_store[key] for key in sorted_keys[:top_k]]

    # Step 4: Cross-Encoder reranking
    pairs        = [(query, chunk) for chunk, _ in rrf_candidates]
    cross_scores = reranker.predict(pairs)
    ranked       = sorted(zip(cross_scores, rrf_candidates),
                          key=lambda x: x[0],
                          reverse=True)

    return [item for _, item in ranked[:final_k]], retrieval_time
  



def generate_answer(query, retrieved_chunks):
    context = "\n\n---\n\n".join([
        f"[Source: {meta.get('title','?')} ({meta.get('year','?')})]\n{chunk}"
        for chunk, meta in retrieved_chunks
    ])
    prompt = f"""You are a research assistant specializing in machine learning.
Answer the question using ONLY the provided context from academic papers.
If the context does not contain enough information, say so explicitly.
Do not fabricate facts or citations.

Context:
{context}

Question: {query}
Answer:"""

    t0       = time.time()
    response = openai_client.chat.completions.create(   # ← changed
        model="gpt-4o-mini",                             # ← changed
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512,
    )
    gen_time = round(time.time() - t0, 3)
    answer   = response.choices[0].message.content

    return answer, context, gen_time





print("all functions defined")



all functions defined


In [11]:
# End-to-end test

print("=" * 60)
print("  END-TO-END PIPELINE TEST")
print("=" * 60)

test_query = "What is Nested Learning and how does it unify deep learning architectures?"

print(f"\nQuery: {test_query}\n")

# Test hybrid retrieval with Strategy B
chunks, ret_time = retrieve(test_query, strategy="B", mode="hybrid")
answer, context, gen_time = generate_answer(test_query, chunks)

print(f"Retrieved {len(chunks)} chunks in {ret_time}s:")
for i, (chunk, meta) in enumerate(chunks):
    print(f"\n  [{i+1}] {meta.get('title','?')[:55]}")
    print(f"       {chunk[:100].strip()}...")

print(f"\n{'='*60}")
print(f"  GENERATED ANSWER")
print(f"{'='*60}")
print(answer)
print(f"\nRetrieval time : {ret_time}s")
print(f"Generation time: {gen_time}s")
print(f"Total time     : {round(ret_time + gen_time, 3)}s")



  END-TO-END PIPELINE TEST

Query: What is Nested Learning and how does it unify deep learning architectures?

Retrieved 5 chunks in 0.744s:

  [1] Nested Learning: The Illusion of Deep Learning Architec
       design challenges in continual learning, architecture design, and computational depth of modern deep...

  [2] Nested Learning: The Illusion of Deep Learning Architec
       methods, and architectures, but it also reveals a new dimension to stacking layers in deep learning...

  [3] Nested Learning: The Illusion of Deep Learning Architec
       Nested Learning: The Illusion of Deep Learning Architecture
Ali Behrouz , Meisam Razaviyayn, Peilin...

  [4] Nested Learning: The Illusion of Deep Learning Architec
       missing pieces of cortex, highlight the brain’s uniform and reusable structure.
Figure 2: Nested Lea...

  [5] Nested Learning: The Illusion of Deep Learning Architec
       to this new NL’s axis, resulting in observing only the solution of optimization problems and so

In [12]:
# Test all 6 configurations quickly
print(f"\n{'='*60}")
print(f"  QUICK TEST — ALL 6 CONFIGURATIONS")
print(f"{'='*60}")

CONFIGS = [
    {"id": 1, "strategy": "A", "mode": "semantic", "label": "Fixed + Semantic Only"},
    {"id": 2, "strategy": "B", "mode": "semantic", "label": "Recursive + Semantic Only"},
    {"id": 3, "strategy": "C", "mode": "semantic", "label": "Semantic + Semantic Only"},
    {"id": 4, "strategy": "A", "mode": "hybrid",   "label": "Fixed + Hybrid + Rerank"},
    {"id": 5, "strategy": "B", "mode": "hybrid",   "label": "Recursive + Hybrid + Rerank"},
    {"id": 6, "strategy": "C", "mode": "hybrid",   "label": "Semantic + Hybrid + Rerank"},
]

print(f"\n  {'Config':<5} {'Label':<35} {'Chunks':>7} {'Ret(s)':>8} {'Gen(s)':>8}")
print(f"  {'-'*65}")

for config in CONFIGS:
    try:
        chunks, ret_time      = retrieve(test_query,
                                         strategy=config["strategy"],
                                         mode=config["mode"])
        answer, _, gen_time   = generate_answer(test_query, chunks)
        status                = "✓"
    except Exception as e:
        ret_time, gen_time    = 0, 0
        chunks                = []
        status                = f"✗ {str(e)[:20]}"

    print(f"  {status} {config['id']:<4} {config['label']:<35} "
          f"{len(chunks):>7} {ret_time:>8} {gen_time:>8}")

print(f"\n{'='*60}")
print(f"  ALL CONFIGS WORKING — Ready for golden set generation")
print(f"{'='*60}")


  QUICK TEST — ALL 6 CONFIGURATIONS

  Config Label                                Chunks   Ret(s)   Gen(s)
  -----------------------------------------------------------------
  ✓ 1    Fixed + Semantic Only                     5    0.781    3.348
  ✓ 2    Recursive + Semantic Only                 5    0.359     3.34
  ✓ 3    Semantic + Semantic Only                  5    0.747    2.315
  ✓ 4    Fixed + Hybrid + Rerank                   5    0.301    3.728
  ✓ 5    Recursive + Hybrid + Rerank               5      0.4     3.27
  ✓ 6    Semantic + Hybrid + Rerank                5    0.292    3.973

  ALL CONFIGS WORKING — Ready for golden set generation
